Play with this -> "qa_dataset_for_rag.jsonl" file
1. Divide these questions into 6 parts based on their origin(file)
2. remove starting 20-30% question and ending 20-30% question from each parts so that I don't any table of content or introduction related unwanted/basic questions. 
3. filter and collect - 1000 questions overall from all the documents 
300 - easy
300 - medium 
400 - hard
4. create a basictest.jsonl file and put those question there to be used in rag system

In [ ]:
# create separate .jsonl files for each of the 6 source_doc types in your dataset

import json

# Define your input file and the list of the 6 allowed source_doc types
input_file_path = "qa_dataset_for_rag.jsonl"
source_doc_types = [ "docs/NIST.CSWP.29.pdf",
    "docs/NIST.SP.800-53r5.pdf",
    "docs/NIST.SP.800-171r3.pdf",
    "docs/OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf",
    "docs/rfc9110.pdf",
    "docs/wellarchitected-framework.pdf"]

# Create a dictionary to hold the open file objects for each type
# This creates files named like: doc_type_1.jsonl, doc_type_2.jsonl, etc.
output_files = {doc_type: open(f"{doc_type}.jsonl", "w", encoding="utf-8") for doc_type in source_doc_types}

# Optional: File to catch any unexpected source_doc types not in your list
fallback_file = open("unmatched_source_docs.jsonl", "w", encoding="utf-8")

try:
    # Read the main file line by line
    with open(input_file_path, "r", encoding="utf-8") as infile:
        for line_number, line in enumerate(infile, 1):
            line = line.strip()
            if not line:
                continue  # Skip empty lines
            
            try:
                # Parse the JSON line into a Python dictionary
                obj = json.loads(line)
                source_type = obj.get("source_doc")
                
                # Check if the source_doc matches one of your 6 types
                if source_type in output_files:
                    output_files[source_type].write(json.dumps(obj) + "\n")
                else:
                    # Write to fallback if the key is missing or unexpected
                    fallback_file.write(json.dumps(obj) + "\n")
                    
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON on line {line_number}")

finally:
    # Crucial step: Safely close all the opened files
    for f in output_files.values():
        f.close()
    fallback_file.close()

print("Splitting complete! Check your directory for the new .jsonl files.")

Splitting complete! Check your directory for the new .jsonl files.


In [25]:
import os

def count_jsonl_objects(file_paths):
    """
    Counts the number of JSON objects in each provided .jsonl file.
    
    :param file_paths: List of strings containing paths to the files.
    :return: A dictionary mapping file names to their respective object counts.
    """
    results = {}
    
    for path in file_paths:
        # Check if the file actually exists before trying to open it
        if not os.path.exists(path):
            print(f"Warning: File not found at {path} Skipping.")
            results[path] = 0
            continue
            
        count = 0
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                # Strip whitespace to avoid counting empty lines at the end of the file
                if line.strip():
                    count += 1
                    
        results[path] = count
        
    return results

In [16]:
# function to count objects in each of the 6 .jsonl files

import json
import os



# --- Example Usage ---

# Define the list of files you want to check
files_to_count = [f"{doc_type}.jsonl" for doc_type in source_doc_types]

# Add the fallback file to the list just in case
files_to_count.append("unmatched_source_docs.jsonl")

# Run the function
file_counts = count_jsonl_objects(files_to_count)

print("\n--- Object Counts per File ---")
for file_name, total_objects in file_counts.items():
    print(f"{file_name}: {total_objects:,} objects")

NameError: name 'source_doc_types' is not defined

In [10]:
tech_documents = ["/docs/NIST.CSWP.29.pdf.jsonl",
                  "/docs/NIST.SP.800-53r5.pdf.jsonl",
                  "/docs/NIST.SP.800-171r3.pdf.jsonl",
                  "/docs/OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf.jsonl",
                  "/docs/rfc9110.pdf.jsonl",
                  "/docs/wellarchitected-framework.pdf.jsonl"]


In [13]:
# remove 20% of the data from starting and ending of each of the 6 .jsonl files and save the remaining 80% in a new file
# in new folder called "processed_data_20_percent_removed"

import os

# 1. Define the input files exactly as they appear in your loop
tech_documents = [
    "/docs/NIST.CSWP.29.pdf.jsonl",
    "/docs/NIST.SP.800-53r5.pdf.jsonl",
    "/docs/NIST.SP.800-171r3.pdf.jsonl",
    "/docs/OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf.jsonl",
    "/docs/rfc9110.pdf.jsonl",
    "/docs/wellarchitected-framework.pdf.jsonl"
]

# 2. Setup the paths relative to where you are running this code
base_dir = os.getcwd()
output_folder_name = "processed_data_20_percent_removed"
output_folder_path = os.path.join(base_dir, output_folder_name)

# Create the new folder if it doesn't exist
os.makedirs(output_folder_path, exist_ok=True)

def process_jsonl_file(input_file_path, output_file_path):
    """
    Processes a .jsonl file by removing 10% from the start and 10% from the end
    (20% total removed), saving the remaining 80% to a new file.
    """
    if not os.path.exists(input_file_path):
        print(f"Warning: File {input_file_path} not found. Skipping.")
        return

    with open(input_file_path, "r", encoding="utf-8") as infile:
        lines = infile.readlines()
        
        total_lines = len(lines)
        if total_lines == 0:
            print(f"Warning: {input_file_path} is empty. No data to process.")
            return
        
        # To keep 80% remaining, remove 10% from the start and 10% from the end
        start_index = int(total_lines * 0.1)
        end_index = int(total_lines * 0.9)  # Trims the last 10%
        
        # Slice the list to get the middle 80%
        processed_lines = lines[start_index:end_index]
        
        # Write the processed lines to the new file
        with open(output_file_path, "w", encoding="utf-8") as outfile:
            outfile.writelines(processed_lines)
            
    print(f"Processed: {os.path.basename(input_file_path)} -> Saved {len(processed_lines):,} lines.")

# --- Main processing loop ---

print("Starting processing script...\n")

for doc_type in tech_documents:
    # Clean up the string: remove the leading slash so os.path.join handles it properly
    # e.g., "/docs/rfc9110.pdf.jsonl" becomes "docs/rfc9110.pdf.jsonl"
    clean_relative_path = doc_type.lstrip("/")
    
    # Build the absolute input path pointing to your existing files
    input_name = os.path.join(base_dir, clean_relative_path)
    
    # Extract just the filename (e.g., rfc9110.pdf.jsonl) to build the output path
    file_name_only = os.path.basename(clean_relative_path)
    output_name = os.path.join(output_folder_path, f"processed_{file_name_only}")
    
    # Execute the slicing function
    process_jsonl_file(input_name, output_name)

print("\nAll files have been successfully processed and saved!")

Starting processing script...

Processed: NIST.CSWP.29.pdf.jsonl -> Saved 265 lines.
Processed: NIST.SP.800-53r5.pdf.jsonl -> Saved 6,257 lines.
Processed: NIST.SP.800-171r3.pdf.jsonl -> Saved 1,032 lines.
Processed: OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf.jsonl -> Saved 910 lines.
Processed: rfc9110.pdf.jsonl -> Saved 1,678 lines.
Processed: wellarchitected-framework.pdf.jsonl -> Saved 7,572 lines.

All files have been successfully processed and saved!


In [27]:
import os

# 1. Your file list containing the leading slashes
processed_data = [
    "/processed_data_20_percent_removed/processed_NIST.CSWP.29.pdf.jsonl",
    "/processed_data_20_percent_removed/processed_NIST.SP.800-53r5.pdf.jsonl",
    "/processed_data_20_percent_removed/processed_NIST.SP.800-171r3.pdf.jsonl",
    "/processed_data_20_percent_removed/processed_OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf.jsonl",
    "/processed_data_20_percent_removed/processed_rfc9110.pdf.jsonl",
    "/processed_data_20_percent_removed/processed_wellarchitected-framework.pdf.jsonl"
]

def count_jsonl_objects(file_paths):
    """
    Counts the number of JSON objects in each provided .jsonl file,
    automatically resolving relative directory structures.
    """
    results = {}
    base_dir = os.getcwd()  # Get current folder where script is executing
    
    for path in file_paths:
        # Crucial fix: strip the leading "/" so os.path.join appends it to your current directory
        clean_relative_path = path.lstrip("/")
        absolute_path = os.path.join(base_dir, clean_relative_path)
        
        # Check if the file actually exists at the resolved path
        if not os.path.exists(absolute_path):
            print(f"Warning: File not found at {absolute_path}. Skipping.")
            results[path] = 0
            continue
            
        count = 0
        with open(absolute_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    count += 1
                    
        results[path] = count
        
    return results

# --- Execution ---

print("Counting lines in processed files...\n")

# Run the fixed function
file_counts = count_jsonl_objects(processed_data)

print("\n--- Object Counts per File ---")
for file_name, total_objects in file_counts.items():
    # Extracts just the final filename to keep the output clean
    short_name = os.path.basename(file_name)
    print(f"{short_name}: {total_objects:,} objects")

Counting lines in processed files...


--- Object Counts per File ---
processed_NIST.CSWP.29.pdf.jsonl: 265 objects
processed_NIST.SP.800-53r5.pdf.jsonl: 6,257 objects
processed_NIST.SP.800-171r3.pdf.jsonl: 1,032 objects
processed_OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf.jsonl: 910 objects
processed_rfc9110.pdf.jsonl: 1,678 objects
processed_wellarchitected-framework.pdf.jsonl: 7,572 objects


In [33]:
# select random 265 questions from each of the 6 processed .jsonl files and save them in a new file called "final_qa_dataset.jsonl" in the current directory

def select_random_questions(input_files, output_file, num_questions=265):
    """
    Selects a specified number of random questions from each input .jsonl file
    and saves them into a single output .jsonl file.
    
    :param input_files: List of input .jsonl file paths.
    :param output_file: Path to the output .jsonl file.
    :param num_questions: Number of random questions to select from each file.
    """
    import random

    with open(output_file, "w", encoding="utf-8") as outfile:
        for input_file in input_files:
            # Resolve the absolute path for each input file
            clean_relative_path = input_file.lstrip("/")
            absolute_path = os.path.join(os.getcwd(), clean_relative_path)
            
            if not os.path.exists(absolute_path):
                print(f"Warning: File not found at {absolute_path}. Skipping.")
                continue
            
            with open(absolute_path, "r", encoding="utf-8") as infile:
                lines = infile.readlines()
                
                if len(lines) < num_questions:
                    print(f"Warning: {absolute_path} has only {len(lines)} lines. Selecting all available lines.")
                    selected_lines = lines
                else:
                    selected_lines = random.sample(lines, num_questions)
                
                # Write the selected lines to the output file
                outfile.writelines(selected_lines)
    
    print(f"Random selection complete! Saved to {output_file}.")
    
    
    
# --- Execution ---
select_random_questions(processed_data, "final_qa_dataset_for_rag.jsonl", 265)

Random selection complete! Saved to final_qa_dataset_for_rag.jsonl.


In [34]:
# count the number of questions in the final_qa_dataset.jsonl file
with open("final_qa_dataset_for_rag.jsonl", "r") as f:
    num_questions = sum(1 for line in f)
print(f"Number of questions: {num_questions}")

Number of questions: 1590


In [36]:
import json
from collections import Counter

def count_difficulty_levels(jsonl_file_path):
    """
    Reads a JSONL file and counts the total number of easy, medium, 
    and hard questions/items.
    """
    # Initialize a counter to keep track of the difficulty frequencies
    difficulty_counts = Counter()
    
    with open(jsonl_file_path, 'r', encoding='utf-8') as file:
        for line_num, line in enumerate(file, 1):
            line = line.strip()
            
            # Skip empty lines
            if not line:
                continue
                
            try:
                # Parse the JSON object from the line
                data = json.loads(line)
                
                # Extract the difficulty key (safely using .get to prevent KeyErrors)
                difficulty = data.get("difficulty")
                
                if difficulty:
                    # Normalize string casing (.lower()) to avoid mismatch (e.g., 'Easy' vs 'easy')
                    difficulty_counts[str(difficulty).lower()] += 1
                else:
                    print(f"Warning: Line {line_num} does not contain a 'difficulty' key.")
                    
            except json.JSONDecodeError:
                print(f"Error: Line {line_num} is not valid JSON. Skipping line.")
                continue

    # Convert the Counter object back into a standard dictionary
    return dict(difficulty_counts)


file_path = "./final_qa_dataset_for_rag.jsonl"
results = count_difficulty_levels(file_path)
print("Difficulty Summary:", results)

Difficulty Summary: {'easy': 1204, 'medium': 376, 'hard': 10}


In [39]:
# we have uneven distribution of easy, medium, and hard questions in the final_qa_dataset_for_rag.jsonl file.
# so lets fix that

processed_data = [
    "./processed_data_20_percent_removed/processed_NIST.CSWP.29.pdf.jsonl",
    "./processed_data_20_percent_removed/processed_NIST.SP.800-53r5.pdf.jsonl",
    "./processed_data_20_percent_removed/processed_NIST.SP.800-171r3.pdf.jsonl",
    "./processed_data_20_percent_removed/processed_OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf.jsonl",
    "./processed_data_20_percent_removed/processed_rfc9110.pdf.jsonl",
    "./processed_data_20_percent_removed/processed_wellarchitected-framework.pdf.jsonl"
]


total_easy = 0
total_medium = 0
total_hard = 0

for doc in processed_data:
    results = count_difficulty_levels(doc)
    print("Difficulty Summary:", results)
    total_easy += results.get("easy", 0)
    total_medium += results.get("medium", 0)
    total_hard += results.get("hard", 0)
   
print("Total Easy:", total_easy)
print("Total Medium:", total_medium)
print("Total Hard:", total_hard)

Difficulty Summary: {'easy': 213, 'medium': 52}
Difficulty Summary: {'medium': 1588, 'easy': 4619, 'hard': 50}
Difficulty Summary: {'easy': 774, 'medium': 252, 'hard': 6}
Difficulty Summary: {'easy': 703, 'medium': 202, 'hard': 5}
Difficulty Summary: {'easy': 1130, 'medium': 513, 'hard': 35}
Difficulty Summary: {'easy': 5887, 'medium': 1644, 'hard': 41}
Total Easy: 13326
Total Medium: 4251
Total Hard: 137


In [40]:
# keep 137 easy, medium and hard questions from each of the 6 processed .jsonl files and save them in a new file called "final_qa_dataset_for_rag_balanced.jsonl" in the current directory

def select_balanced_questions(input_files, output_file, num_per_difficulty=137):
    """
    Selects a specified number of questions for each difficulty level (easy, medium, hard)
    from each input .jsonl file and saves them into a single output .jsonl file.
    
    :param input_files: List of input .jsonl file paths.
    :param output_file: Path to the output .jsonl file.
    :param num_per_difficulty: Number of questions to select for each difficulty level from each file.
    """
    import random

    with open(output_file, "w", encoding="utf-8") as outfile:
        for input_file in input_files:
            # Resolve the absolute path for each input file
            clean_relative_path = input_file.lstrip("/")
            absolute_path = os.path.join(os.getcwd(), clean_relative_path)
            
            if not os.path.exists(absolute_path):
                print(f"Warning: File not found at {absolute_path}. Skipping.")
                continue
            
            # Initialize a dictionary to hold questions by difficulty
            questions_by_difficulty = {"easy": [], "medium": [], "hard": []}
            
            with open(absolute_path, "r", encoding="utf-8") as infile:
                for line in infile:
                    line = line.strip()
                    if not line:
                        continue
                    
                    try:
                        data = json.loads(line)
                        difficulty = data.get("difficulty", "").lower()
                        
                        if difficulty in questions_by_difficulty:
                            questions_by_difficulty[difficulty].append(line)
                    except json.JSONDecodeError:
                        print(f"Error: Invalid JSON in {absolute_path}. Skipping line.")
                        continue
            
            # Randomly sample the specified number of questions for each difficulty
            for difficulty, questions in questions_by_difficulty.items():
                if len(questions) < num_per_difficulty:
                    print(f"Warning: {absolute_path} has only {len(questions)} '{difficulty}' questions. Selecting all available.")
                    selected_questions = questions
                else:
                    selected_questions = random.sample(questions, num_per_difficulty)
                
                # Write the selected questions to the output file
                outfile.writelines(selected_questions)
    
    print(f"Balanced selection complete! Saved to {output_file}.")
    

select_balanced_questions(processed_data, "final_qa_dataset_for_rag_balanced.jsonl", 137)

Balanced selection complete! Saved to final_qa_dataset_for_rag_balanced.jsonl.
